In [ ]:
import importlib
import sys

# Force reload of helpers module to pick up latest changes
if 'notebooks.helpers' in sys.modules:
    importlib.reload(sys.modules['notebooks.helpers'])
    importlib.reload(sys.modules['notebooks.helpers.database'])
    importlib.reload(sys.modules['notebooks.helpers.logging_config'])


In [0]:
# ==============================================================================
# CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================

from notebooks.helpers import (
    write_gold_table, setup_logger,
    transform_service_fact, transform_rental_fact,
    BronzeLoader
)
import time

# Setup logging
logger = setup_logger("load_facts")

logger.info("=" * 70)
logger.info("FACT LOAD JOB STARTED")
logger.info("=" * 70)

WATERMARK_COLUMNS = {
    "service": "service_date",
    "rental": "rental_date",
    "staff": "last_update",
    "inventory": "last_update",
    "payment": "last_update",
}
bronze_loader = BronzeLoader(spark, dbutils)


def load_and_persist_bronze(table_name: str):
    """Full-load raw table into bronze and keep watermarks in sync."""
    watermark_column = WATERMARK_COLUMNS.get(table_name, "last_update")
    return bronze_loader.load_full_to_bronze(
        table_name=table_name,
        watermark_column=watermark_column,
        updated_by="load_all_facts",
    )


In [0]:
# ==============================================================================
# BRONZE LAYER: Load source tables (persisted to bronze)
# ==============================================================================

service_bronze = load_and_persist_bronze("service")
rental_bronze = load_and_persist_bronze("rental")
staff_bronze = load_and_persist_bronze("staff")
inventory_bronze = load_and_persist_bronze("inventory")
payment_bronze = load_and_persist_bronze("payment")


In [0]:
# ==============================================================================
# GOLD: FACT_SERVICE
# ==============================================================================

logger.info("GOLD: Building fact_service")
start_time = time.time()

fact_service = transform_service_fact(service_bronze)

write_gold_table(fact_service, "fact_service", mode="overwrite", partition_by=["service_date"])
logger.info(f"GOLD: fact_service completed in {time.time() - start_time:.2f}s")

In [ ]:
# ==============================================================================
# GOLD: FACT_RENTAL
# ==============================================================================

logger.info("GOLD: Building fact_rental")
start_time = time.time()

fact_rental = transform_rental_fact(
    rental_bronze, staff_bronze, inventory_bronze, payment_bronze
)

write_gold_table(fact_rental, "fact_rental", mode="overwrite", partition_by=["rental_date"])
logger.info(f"GOLD: fact_rental completed in {time.time() - start_time:.2f}s")

# ==============================================================================
# JOB COMPLETION
# ==============================================================================
logger.info("=" * 70)
logger.info("FACT LOAD JOB COMPLETED SUCCESSFULLY")
logger.info("=" * 70)